In [160]:
!pip install numpy pandas scipy scikit-learn tensorflow matplotlib

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score
)

import tensorflow as tf
from google.colab import files

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
WINDOW_SIZE = 128

CLASS_FILES = {
    "circle": "Circle.csv",
    "left_right": "Left-right.csv",
    "rest": "Rest.csv",
    "up_down": "Up-down.csv"
}

In [ ]:
import os

for label, fname in CLASS_FILES.items():
    print(label, "->", fname, "exists:", os.path.exists(fname))

circle -> Circle.csv exists: True
left_right -> Left-right.csv exists: True
rest -> Rest.csv exists: True
up_down -> Up-down.csv exists: True


In [ ]:
def load_windows_from_csv(filepath, window_size=128):
    df = pd.read_csv(filepath)

    # Handle CSVs stored as a single comma-separated column
    if len(df.columns) == 1:
        col = df.columns[0]
        df = df[col].str.split(",", expand=True)

    # We expect exactly 6 IMU channels
    if df.shape[1] != 6:
        raise ValueError(
            f"{filepath}: expected 6 columns, got {df.shape[1]}"
        )

    df.columns = ["aX", "aY", "aZ", "gX", "gY", "gZ"]

    # Convert everything to numeric and remove invalid rows
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna().reset_index(drop=True)

    values = df.to_numpy(dtype=np.float32)

    # Keep only complete non-overlapping windows
    num_windows = len(values) // window_size
    usable_samples = num_windows * window_size
    dropped_samples = len(values) - usable_samples

    values = values[:usable_samples]
    windows = values.reshape(num_windows, window_size, 6)

    print(
        f"{filepath}: "
        f"{len(df)} valid rows -> "
        f"{num_windows} windows, "
        f"{dropped_samples} rows dropped"
    )

    return windows

In [ ]:
all_data = {}

for label, filepath in CLASS_FILES.items():
    windows = load_windows_from_csv(filepath, WINDOW_SIZE)
    all_data[label] = windows
    print(label, windows.shape)

Circle.csv: 10240 valid rows -> 80 windows, 0 rows dropped
circle (80, 128, 6)
Left-right.csv: 10240 valid rows -> 80 windows, 0 rows dropped
left_right (80, 128, 6)
Rest.csv: 10240 valid rows -> 80 windows, 0 rows dropped
rest (80, 128, 6)
Up-down.csv: 10240 valid rows -> 80 windows, 0 rows dropped
up_down (80, 128, 6)


In [ ]:
def extract_features_from_window(window, fs=100):
    """
    Extract combined time-domain and frequency-domain features
    from one IMU window.

    Input shape:
        (128, 6)

    Features per axis:
        Time domain:
            1. Mean
            2. Standard deviation
            3. RMS
            4. Minimum
            5. Maximum

        Frequency domain:
            6. Dominant frequency
            7. Peak PSD

    Total:
        7 features × 6 axes = 42 features
    """

    features = []

    for axis in range(window.shape[1]):

        signal = window[:, axis]

        # ==========================================
        # Time-domain features
        # ==========================================

        mean_val = np.mean(signal)
        std_val = np.std(signal)
        rms_val = np.sqrt(np.mean(signal ** 2))
        min_val = np.min(signal)
        max_val = np.max(signal)

        # ==========================================
        # Frequency-domain features
        # ==========================================

        freqs, psd = welch(
            signal,
            fs=fs,
            nperseg=min(len(signal), 64)
        )

        dominant_freq = freqs[np.argmax(psd)]
        peak_psd = np.max(psd)

        # ==========================================
        # Combined feature vector
        # ==========================================

        features.extend([
            mean_val,
            std_val,
            rms_val,
            min_val,
            max_val,
            dominant_freq,
            peak_psd
        ])

    return np.asarray(features, dtype=np.float32)

In [ ]:
def extract_features_configurable(window, mode="combined", fs=100):
    """
    Feature extraction for TinyML gesture classification.

    Modes:
        time:
            Mean, Std, RMS, Min, Max
            5 features per axis -> 30 total

        frequency:
            Dominant Frequency, Peak PSD
            2 features per axis -> 12 total

        combined:
            Time + Frequency
            7 features per axis -> 42 total

        tiny:
            Std, RMS, Max
            3 features per axis -> 18 total
    """

    features = []

    for axis in range(window.shape[1]):

        signal = window[:, axis]

        # ==========================
        # Time-domain features
        # ==========================

        mean_val = np.mean(signal)
        std_val = np.std(signal)
        rms_val = np.sqrt(np.mean(signal ** 2))
        min_val = np.min(signal)
        max_val = np.max(signal)

        # ==========================
        # Frequency-domain features
        # ==========================

        freqs, psd = welch(
            signal,
            fs=fs,
            nperseg=min(len(signal), 64)
        )

        dominant_freq = freqs[np.argmax(psd)]
        peak_psd = np.max(psd)

        # ==========================
        # Select feature configuration
        # ==========================

        if mode == "time":

            axis_features = [
                mean_val,
                std_val,
                rms_val,
                min_val,
                max_val
            ]

        elif mode == "frequency":

            axis_features = [
                dominant_freq,
                peak_psd
            ]

        elif mode == "combined":

            axis_features = [
                mean_val,
                std_val,
                rms_val,
                min_val,
                max_val,
                dominant_freq,
                peak_psd
            ]

        elif mode == "tiny":

            axis_features = [
                std_val,
                rms_val,
                max_val
            ]

        else:
            raise ValueError(
                "mode must be one of: "
                "'time', 'frequency', 'combined', 'tiny'"
            )

        features.extend(axis_features)

    return np.asarray(features, dtype=np.float32)

In [ ]:
sample_window = all_data["circle"][0]

print(
    "Time:",
    extract_features_configurable(
        sample_window,
        mode="time"
    ).shape
)

print(
    "Frequency:",
    extract_features_configurable(
        sample_window,
        mode="frequency"
    ).shape
)

print(
    "Combined:",
    extract_features_configurable(
        sample_window,
        mode="combined"
    ).shape
)

print(
    "Tiny:",
    extract_features_configurable(
        sample_window,
        mode="tiny"
    ).shape
)

Time: (30,)
Frequency: (12,)
Combined: (42,)
Tiny: (18,)


In [ ]:
class_names = ["circle", "left_right", "rest", "up_down"]

In [ ]:
feature_modes = [
    "time",
    "frequency",
    "combined",
    "tiny"
]

datasets = {}

for mode in feature_modes:

    X = []
    y = []

    for label_idx, label in enumerate(class_names):

        for window in all_data[label]:

            features = extract_features_configurable(
                window,
                mode=mode,
                fs=100
            )

            X.append(features)
            y.append(label_idx)

    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int32)

    datasets[mode] = {
        "X": X,
        "y": y
    }

    print(
        f"{mode.upper():10s} -> "
        f"X shape: {X.shape}, "
        f"y shape: {y.shape}"
    )

TIME       -> X shape: (320, 30), y shape: (320,)
FREQUENCY  -> X shape: (320, 12), y shape: (320,)
COMBINED   -> X shape: (320, 42), y shape: (320,)
TINY       -> X shape: (320, 18), y shape: (320,)


In [ ]:
# ============================================================
# Shared Train / Validation / Test Split
# Same indices will be used for ALL feature configurations
# ============================================================

y = datasets["combined"]["y"]
indices = np.arange(len(y))

# First: reserve 20% for final test
train_val_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# Then: take 25% of remaining 80% = 20% of total for validation
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.25,
    random_state=SEED,
    stratify=y[train_val_idx]
)

print("Total samples:     ", len(indices))
print("Train samples:     ", len(train_idx))
print("Validation samples:", len(val_idx))
print("Test samples:      ", len(test_idx))

print()
print("Train ratio:      ", len(train_idx) / len(indices))
print("Validation ratio: ", len(val_idx) / len(indices))
print("Test ratio:       ", len(test_idx) / len(indices))

Total samples:      320
Train samples:      192
Validation samples: 64
Test samples:       64

Train ratio:       0.6
Validation ratio:  0.2
Test ratio:        0.2


In [ ]:
from sklearn.preprocessing import StandardScaler

prepared_data = {}
scalers = {}

for mode in ["time", "frequency", "combined", "tiny"]:

    X = datasets[mode]["X"]
    y = datasets[mode]["y"]

    X_train = X[train_idx]
    y_train = y[train_idx]

    X_val = X[val_idx]
    y_val = y[val_idx]

    X_test = X[test_idx]
    y_test = y[test_idx]

    scaler = StandardScaler()

    # Fit ONLY on training data
    X_train_scaled = scaler.fit_transform(X_train)

    # Apply the same scaler to validation and test
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    scalers[mode] = scaler

    prepared_data[mode] = {
        "X_train": X_train_scaled.astype(np.float32),
        "y_train": y_train,

        "X_val": X_val_scaled.astype(np.float32),
        "y_val": y_val,

        "X_test": X_test_scaled.astype(np.float32),
        "y_test": y_test
    }

    print(
        f"{mode.upper():10s} -> "
        f"Train: {X_train_scaled.shape}, "
        f"Val: {X_val_scaled.shape}, "
        f"Test: {X_test_scaled.shape}"
    )

TIME       -> Train: (192, 30), Val: (64, 30), Test: (64, 30)
FREQUENCY  -> Train: (192, 12), Val: (64, 12), Test: (64, 12)
COMBINED   -> Train: (192, 42), Val: (64, 42), Test: (64, 42)
TINY       -> Train: (192, 18), Val: (64, 18), Test: (64, 18)


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

def build_dense_model(input_dim, num_classes=4):
    model = Sequential([
        tf.keras.Input(shape=(input_dim,)),

        Dense(
            64,
            activation="relu",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED)
        ),

        Dense(
            32,
            activation="relu",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED)
        ),

        Dense(
            num_classes,
            activation="softmax",
            kernel_initializer=tf.keras.initializers.GlorotUniform(seed=SEED)
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


print("build_dense_model is ready.")

build_dense_model is ready.


In [ ]:
models = {}
histories = {}
test_results = {}

for mode in ["time", "frequency", "combined", "tiny"]:

    print("\n" + "=" * 60)
    print(f"Training DENSE model with {mode.upper()} features")
    print("=" * 60)

    data = prepared_data[mode]

    # Reproducibility before building each model
    tf.keras.utils.set_random_seed(SEED)

    model = build_dense_model(
        input_dim=data["X_train"].shape[1],
        num_classes=len(class_names)
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )

    history = model.fit(
        data["X_train"],
        data["y_train"],
        validation_data=(
            data["X_val"],
            data["y_val"]
        ),
        epochs=150,
        batch_size=16,
        callbacks=[early_stop],
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        data["X_test"],
        data["y_test"],
        verbose=0
    )

    models[mode] = model
    histories[mode] = history

    test_results[mode] = {
        "loss": float(test_loss),
        "accuracy": float(test_accuracy),
        "epochs": len(history.history["loss"]),
        "params": model.count_params()
    }

    print(f"Input features: {data['X_train'].shape[1]}")
    print(f"Parameters:     {model.count_params():,}")
    print(f"Epochs:         {len(history.history['loss'])}")
    print(f"Test loss:      {test_loss:.4f}")
    print(f"Test accuracy:  {test_accuracy:.4f}")


Training DENSE model with TIME features
Input features: 30
Parameters:     4,196
Epochs:         150
Test loss:      0.0001
Test accuracy:  1.0000

Training DENSE model with FREQUENCY features
Input features: 12
Parameters:     3,044
Epochs:         150
Test loss:      0.0406
Test accuracy:  0.9844

Training DENSE model with COMBINED features
Input features: 42
Parameters:     4,964
Epochs:         150
Test loss:      0.0012
Test accuracy:  1.0000

Training DENSE model with TINY features
Input features: 18
Parameters:     3,428
Epochs:         150
Test loss:      0.0001
Test accuracy:  1.0000


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

comparison_results = {}

for mode in ["time", "frequency", "combined", "tiny"]:

    model = models[mode]
    data = prepared_data[mode]

    y_prob = model.predict(
        data["X_test"],
        verbose=0
    )

    y_pred = np.argmax(
        y_prob,
        axis=1
    )

    f1 = f1_score(
        data["y_test"],
        y_pred,
        average="macro"
    )

    params = model.count_params()

    comparison_results[mode] = {
        "accuracy": float(test_results[mode]["accuracy"]),
        "f1_macro": float(f1),
        "params": int(params),
        "epochs": int(test_results[mode]["epochs"])
    }

    print("\n" + "=" * 60)
    print(f"{mode.upper()} FEATURES")
    print("=" * 60)

    print(
        f"Accuracy:   "
        f"{comparison_results[mode]['accuracy']:.4f}"
    )

    print(
        f"Macro F1:   "
        f"{comparison_results[mode]['f1_macro']:.4f}"
    )

    print(
        f"Parameters: "
        f"{params:,}"
    )

    print(
        f"Epochs:     "
        f"{test_results[mode]['epochs']}"
    )

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            data["y_test"],
            y_pred
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            data["y_test"],
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )


TIME FEATURES
Accuracy:   1.0000
Macro F1:   1.0000
Parameters: 4,196
Epochs:     150

Confusion Matrix:
[[16  0  0  0]
 [ 0 16  0  0]
 [ 0  0 16  0]
 [ 0  0  0 16]]

Classification Report:
              precision    recall  f1-score   support

      circle     1.0000    1.0000    1.0000        16
  left_right     1.0000    1.0000    1.0000        16
        rest     1.0000    1.0000    1.0000        16
     up_down     1.0000    1.0000    1.0000        16

    accuracy                         1.0000        64
   macro avg     1.0000    1.0000    1.0000        64
weighted avg     1.0000    1.0000    1.0000        64


FREQUENCY FEATURES
Accuracy:   0.9844
Macro F1:   0.9844
Parameters: 3,044
Epochs:     150

Confusion Matrix:
[[16  0  0  0]
 [ 1 15  0  0]
 [ 0  0 16  0]
 [ 0  0  0 16]]

Classification Report:
              precision    recall  f1-score   support

      circle     0.9412    1.0000    0.9697        16
  left_right     1.0000    0.9375    0.9677        16
        rest   

In [ ]:
import pandas as pd

feature_counts = {
    "time": 30,
    "frequency": 12,
    "combined": 42,
    "tiny": 18
}

comparison_df = pd.DataFrame(comparison_results).T

comparison_df["num_features"] = (
    comparison_df.index
    .map(feature_counts)
)

comparison_df = comparison_df[
    [
        "num_features",
        "params",
        "accuracy",
        "f1_macro",
        "epochs"
    ]
]

comparison_df = comparison_df.sort_values(
    by="num_features"
)

comparison_df

,num_features,params,accuracy,f1_macro,epochs
frequency,12,3044.0,0.984375,0.98436,150.0
tiny,18,3428.0,1.000000,1.00000,150.0
time,30,4196.0,1.000000,1.00000,150.0
combined,42,4964.0,1.000000,1.00000,150.0


In [ ]:
def build_optimized_dense_model(mode, num_classes=4):

    if mode == "combined":
        input_dim = 42
        hidden1 = 64
        hidden2 = 32

    elif mode == "time":
        input_dim = 30
        hidden1 = 48
        hidden2 = 24

    elif mode == "tiny":
        input_dim = 18
        hidden1 = 16
        hidden2 = 8

    elif mode == "frequency":
        input_dim = 12
        hidden1 = 16
        hidden2 = 8

    else:
        raise ValueError(
            "mode must be one of: "
            "'combined', 'time', 'frequency', 'tiny'"
        )

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(input_dim,)),

        tf.keras.layers.Dense(
            hidden1,
            activation="relu",
            kernel_initializer=tf.keras.initializers.GlorotUniform(
                seed=SEED
            )
        ),

        tf.keras.layers.Dense(
            hidden2,
            activation="relu",
            kernel_initializer=tf.keras.initializers.GlorotUniform(
                seed=SEED
            )
        ),

        tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
            kernel_initializer=tf.keras.initializers.GlorotUniform(
                seed=SEED
            )
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
for mode in ["combined", "time", "frequency", "tiny"]:

    model = build_optimized_dense_model(
        mode,
        num_classes=len(class_names)
    )

    print(
        f"{mode.upper():10s} -> "
        f"Parameters: {model.count_params():,}"
    )

COMBINED   -> Parameters: 4,964
TIME       -> Parameters: 2,764
FREQUENCY  -> Parameters: 380
TINY       -> Parameters: 476


In [ ]:
optimized_models = {}
optimized_histories = {}
optimized_results = {}

for mode in ["combined", "time", "frequency", "tiny"]:

    print("\n" + "=" * 60)
    print(f"Training OPTIMIZED {mode.upper()} model")
    print("=" * 60)

    data = prepared_data[mode]

    # Reproducibility
    tf.keras.utils.set_random_seed(SEED)

    model = build_optimized_dense_model(
        mode,
        num_classes=len(class_names)
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )

    history = model.fit(
        data["X_train"],
        data["y_train"],
        validation_data=(
            data["X_val"],
            data["y_val"]
        ),
        epochs=150,
        batch_size=16,
        callbacks=[early_stop],
        verbose=0
    )

    test_loss, test_accuracy = model.evaluate(
        data["X_test"],
        data["y_test"],
        verbose=0
    )

    optimized_models[mode] = model
    optimized_histories[mode] = history

    optimized_results[mode] = {
        "loss": float(test_loss),
        "accuracy": float(test_accuracy),
        "epochs": len(history.history["loss"]),
        "params": model.count_params()
    }

    print(f"Input features: {data['X_train'].shape[1]}")
    print(f"Parameters:     {model.count_params():,}")
    print(f"Epochs:         {len(history.history['loss'])}")
    print(f"Test loss:      {test_loss:.4f}")
    print(f"Test accuracy:  {test_accuracy:.4f}")


Training OPTIMIZED COMBINED model
Input features: 42
Parameters:     4,964
Epochs:         150
Test loss:      0.0012
Test accuracy:  1.0000

Training OPTIMIZED TIME model
Input features: 30
Parameters:     2,764
Epochs:         150
Test loss:      0.0009
Test accuracy:  1.0000

Training OPTIMIZED FREQUENCY model
Input features: 12
Parameters:     380
Epochs:         150
Test loss:      0.0265
Test accuracy:  0.9844

Training OPTIMIZED TINY model
Input features: 18
Parameters:     476
Epochs:         150
Test loss:      0.0005
Test accuracy:  1.0000


In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score
)

optimized_comparison = {}

for mode in ["combined", "time", "frequency", "tiny"]:

    model = optimized_models[mode]
    data = prepared_data[mode]

    y_prob = model.predict(
        data["X_test"],
        verbose=0
    )

    y_pred = np.argmax(
        y_prob,
        axis=1
    )

    f1 = f1_score(
        data["y_test"],
        y_pred,
        average="macro"
    )

    optimized_comparison[mode] = {
        "features": int(data["X_test"].shape[1]),
        "params": int(model.count_params()),
        "accuracy": float(
            optimized_results[mode]["accuracy"]
        ),
        "f1_macro": float(f1),
        "epochs": int(
            optimized_results[mode]["epochs"]
        )
    }

    print("\n" + "=" * 60)
    print(f"OPTIMIZED {mode.upper()} MODEL")
    print("=" * 60)

    print(
        f"Features:   "
        f"{data['X_test'].shape[1]}"
    )

    print(
        f"Parameters: "
        f"{model.count_params():,}"
    )

    print(
        f"Accuracy:   "
        f"{optimized_results[mode]['accuracy']:.4f}"
    )

    print(
        f"Macro F1:   "
        f"{f1:.4f}"
    )

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            data["y_test"],
            y_pred
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            data["y_test"],
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )


OPTIMIZED COMBINED MODEL
Features:   42
Parameters: 4,964
Accuracy:   1.0000
Macro F1:   1.0000

Confusion Matrix:
[[16  0  0  0]
 [ 0 16  0  0]
 [ 0  0 16  0]
 [ 0  0  0 16]]

Classification Report:
              precision    recall  f1-score   support

      circle     1.0000    1.0000    1.0000        16
  left_right     1.0000    1.0000    1.0000        16
        rest     1.0000    1.0000    1.0000        16
     up_down     1.0000    1.0000    1.0000        16

    accuracy                         1.0000        64
   macro avg     1.0000    1.0000    1.0000        64
weighted avg     1.0000    1.0000    1.0000        64


OPTIMIZED TIME MODEL
Features:   30
Parameters: 2,764
Accuracy:   1.0000
Macro F1:   1.0000

Confusion Matrix:
[[16  0  0  0]
 [ 0 16  0  0]
 [ 0  0 16  0]
 [ 0  0  0 16]]

Classification Report:
              precision    recall  f1-score   support

      circle     1.0000    1.0000    1.0000        16
  left_right     1.0000    1.0000    1.0000        16
    

In [ ]:
import tensorflow as tf
import os

tflite_float_models = {}

for mode in ["combined", "time", "frequency", "tiny"]:

    model = optimized_models[mode]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    tflite_model = converter.convert()

    filename = f"{mode}_float32.tflite"

    with open(filename, "wb") as f:
        f.write(tflite_model)

    size_bytes = os.path.getsize(filename)
    size_kb = size_bytes / 1024

    tflite_float_models[mode] = {
        "model": tflite_model,
        "filename": filename,
        "size_bytes": size_bytes,
        "size_kb": size_kb
    }

    print(
        f"{mode.upper():10s} -> "
        f"{filename} | "
        f"{size_kb:.2f} KB"
    )

Saved artifact at '/tmp/tmpgdfviaxe'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='keras_tensor_166')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137588643996368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076977104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076971344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076974416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076977680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076977872: TensorSpec(shape=(), dtype=tf.resource, name=None)
COMBINED   -> combined_float32.tflite | 21.51 KB
Saved artifact at '/tmp/tmpavhmgyo2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30), dtype=tf.float32, name='keras_tensor_170')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.flo

In [ ]:
def representative_dataset(mode, num_samples=100):
    X_train = prepared_data[mode]["X_train"]

    rng = np.random.default_rng(SEED)

    sample_count = min(
        num_samples,
        len(X_train)
    )

    selected_indices = rng.choice(
        len(X_train),
        size=sample_count,
        replace=False
    )

    for idx in selected_indices:
        sample = X_train[idx:idx + 1].astype(np.float32)
        yield [sample]

In [ ]:
tflite_int8_models = {}

for mode in ["combined", "time", "frequency", "tiny"]:

    model = optimized_models[mode]

    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Enable post-training quantization
    converter.optimizations = [
        tf.lite.Optimize.DEFAULT
    ]

    # Representative training samples for calibration
    converter.representative_dataset = (
        lambda mode=mode: representative_dataset(mode)
    )

    # Force fully INT8 operators
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8
    ]

    # INT8 input and output
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    # Convert
    tflite_model = converter.convert()

    filename = f"{mode}_int8.tflite"

    with open(filename, "wb") as f:
        f.write(tflite_model)

    size_bytes = os.path.getsize(filename)
    size_kb = size_bytes / 1024

    tflite_int8_models[mode] = {
        "model": tflite_model,
        "filename": filename,
        "size_bytes": size_bytes,
        "size_kb": size_kb
    }

    print(
        f"{mode.upper():10s} -> "
        f"{filename} | "
        f"{size_kb:.2f} KB"
    )

Saved artifact at '/tmp/tmpt_6g25ml'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 42), dtype=tf.float32, name='keras_tensor_166')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137588643996368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076977104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076971344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076974416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076977680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076977872: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


COMBINED   -> combined_int8.tflite | 9.88 KB
Saved artifact at '/tmp/tmpc8ikk0p6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 30), dtype=tf.float32, name='keras_tensor_170')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137588655669328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588635424080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588655673168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588655677008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588655670288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588631559952: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


TIME       -> time_int8.tflite | 7.10 KB
Saved artifact at '/tmp/tmpeb0z33qf'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='keras_tensor_174')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137588631568208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588631561680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588631563408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588631556688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588631566096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076983632: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


FREQUENCY  -> frequency_int8.tflite | 3.51 KB
Saved artifact at '/tmp/tmp89ad9ne5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 18), dtype=tf.float32, name='keras_tensor_178')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137589076976528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076971920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076981136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076973648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076983248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137589076982672: TensorSpec(shape=(), dtype=tf.resource, name=None)
TINY       -> tiny_int8.tflite | 3.60 KB


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
for mode in ["full", "medium", "tiny"]:
    float_size = tflite_float_models[mode]["size_kb"]
    int8_size = tflite_int8_models[mode]["size_kb"]

    reduction = 100 * (1 - int8_size / float_size)

    print(
        f"{mode.upper():6s} | "
        f"Float32: {float_size:.2f} KB | "
        f"INT8: {int8_size:.2f} KB | "
        f"Reduction: {reduction:.1f}%"
    )

KeyError: 'full'

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate_int8_tflite(
    model_content,
    X_test,
    y_test
):

    interpreter = tf.lite.Interpreter(
        model_content=model_content
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = (
        input_details["quantization"]
    )

    output_scale, output_zero_point = (
        output_details["quantization"]
    )

    print("Input dtype:", input_details["dtype"])
    print("Output dtype:", output_details["dtype"])

    print(
        "Input quantization:",
        input_scale,
        input_zero_point
    )

    print(
        "Output quantization:",
        output_scale,
        output_zero_point
    )

    if input_scale == 0:
        raise ValueError(
            "Invalid input quantization scale = 0"
        )

    if output_scale == 0:
        raise ValueError(
            "Invalid output quantization scale = 0"
        )

    predictions = []

    for sample in X_test:

        sample = (
            sample
            .reshape(1, -1)
            .astype(np.float32)
        )

        # ==========================
        # Float -> INT8
        # ==========================

        sample_quantized = (
            sample / input_scale
            + input_zero_point
        )

        sample_quantized = np.clip(
            np.round(sample_quantized),
            -128,
            127
        ).astype(np.int8)

        interpreter.set_tensor(
            input_details["index"],
            sample_quantized
        )

        interpreter.invoke()

        output = interpreter.get_tensor(
            output_details["index"]
        )

        # ==========================
        # INT8 -> Float
        # ==========================

        output_float = (
            (
                output.astype(np.float32)
                - output_zero_point
            )
            * output_scale
        )

        predictions.append(
            np.argmax(output_float[0])
        )

    predictions = np.asarray(
        predictions,
        dtype=np.int32
    )

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return float(accuracy), predictions

In [ ]:
int8_results = {}

for mode in ["combined", "time", "frequency", "tiny"]:

    print("\n" + "=" * 60)
    print(f"INT8 {mode.upper()}")
    print("=" * 60)

    accuracy, predictions = evaluate_int8_tflite(
        tflite_int8_models[mode]["model"],
        prepared_data[mode]["X_test"],
        prepared_data[mode]["y_test"]
    )

    f1 = f1_score(
        prepared_data[mode]["y_test"],
        predictions,
        average="macro"
    )

    int8_results[mode] = {
        "accuracy": float(accuracy),
        "f1_macro": float(f1),
        "predictions": predictions
    }

    print(f"INT8 Test Accuracy: {accuracy:.4f}")
    print(f"INT8 Macro F1:      {f1:.4f}")

In [ ]:
import pandas as pd

final_results = []

for mode in ["combined", "time", "frequency", "tiny"]:

    final_results.append({
        "model": mode,
        "features": prepared_data[mode]["X_train"].shape[1],
        "parameters": optimized_models[mode].count_params(),

        "float32_size_kb":
            tflite_float_models[mode]["size_kb"],

        "int8_size_kb":
            tflite_int8_models[mode]["size_kb"],

        "float_accuracy":
            optimized_results[mode]["accuracy"],

        "int8_accuracy":
            int8_results[mode]["accuracy"],

        "float_macro_f1":
            optimized_comparison[mode]["f1_macro"],

        "int8_macro_f1":
            int8_results[mode]["f1_macro"]
    })


final_benchmark_df = pd.DataFrame(final_results)

# ============================================================
# Baseline = COMBINED feature model
# ============================================================

baseline_params = final_benchmark_df.loc[
    final_benchmark_df["model"] == "combined",
    "parameters"
].iloc[0]

baseline_features = final_benchmark_df.loc[
    final_benchmark_df["model"] == "combined",
    "features"
].iloc[0]

# ============================================================
# Feature reduction vs Combined
# ============================================================

final_benchmark_df["feature_reduction_vs_combined_%"] = (
    100 * (
        1 -
        final_benchmark_df["features"]
        / baseline_features
    )
)

# ============================================================
# Parameter reduction vs Combined
# ============================================================

final_benchmark_df["parameter_reduction_vs_combined_%"] = (
    100 * (
        1 -
        final_benchmark_df["parameters"]
        / baseline_params
    )
)

# ============================================================
# Model size reduction from Float32 -> INT8
# ============================================================

final_benchmark_df["quantization_size_reduction_%"] = (
    100 * (
        1 -
        final_benchmark_df["int8_size_kb"]
        / final_benchmark_df["float32_size_kb"]
    )
)

# ============================================================
# Accuracy change after quantization
# Positive value = accuracy drop
# Negative value = INT8 slightly better
# ============================================================

final_benchmark_df["accuracy_drop_after_int8"] = (
    final_benchmark_df["float_accuracy"]
    -
    final_benchmark_df["int8_accuracy"]
)

# ============================================================
# F1 change after quantization
# ============================================================

final_benchmark_df["f1_drop_after_int8"] = (
    final_benchmark_df["float_macro_f1"]
    -
    final_benchmark_df["int8_macro_f1"]
)

final_benchmark_df

In [ ]:
final_benchmark_df.to_csv(
    "offline_model_benchmark.csv",
    index=False
)

print("Saved: offline_model_benchmark.csv")

In [ ]:
import matplotlib.pyplot as plt

plot_df = final_benchmark_df.copy()

plot_df["model_label"] = plot_df["model"].map({
    "combined": "Combined",
    "time": "Time",
    "frequency": "Frequency",
    "tiny": "Tiny"
})

plt.figure(figsize=(7, 4))

plt.bar(
    plot_df["model_label"],
    plot_df["parameters"]
)

plt.ylabel("Number of Parameters")
plt.xlabel("Feature Configuration")
plt.title("Parameter Count of Optimized Dense Models")

plt.tight_layout()
plt.show()

In [ ]:
x = range(len(final_benchmark_df))

model_labels = final_benchmark_df["model"].map({
    "combined": "Combined",
    "time": "Time",
    "frequency": "Frequency",
    "tiny": "Tiny"
})

plt.figure(figsize=(7, 4))

plt.bar(
    [i - 0.2 for i in x],
    final_benchmark_df["float32_size_kb"],
    width=0.4,
    label="Float32"
)

plt.bar(
    [i + 0.2 for i in x],
    final_benchmark_df["int8_size_kb"],
    width=0.4,
    label="INT8"
)

plt.xticks(
    list(x),
    model_labels
)

plt.xlabel("Feature Configuration")
plt.ylabel("TFLite Model Size (KB)")
plt.title("Float32 vs INT8 TFLite Model Size")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OPTIONAL DEPLOYMENT SECTION
# Convert selected INT8 model to C header for Arduino
# ============================================================

In [ ]:
# Convert Tiny INT8 TFLite model to C header
# OPTIONAL: only needed for Arduino deployment

model_bytes = tflite_int8_models["tiny"]["model"]

with open("tiny_model.h", "w") as f:
    f.write("#ifndef TINY_MODEL_H\n")
    f.write("#define TINY_MODEL_H\n\n")

    f.write("const unsigned char tiny_model[] = {\n")

    for i, byte in enumerate(model_bytes):
        if i % 12 == 0:
            f.write("    ")

        f.write(f"0x{byte:02x}")

        if i != len(model_bytes) - 1:
            f.write(", ")

        if (i + 1) % 12 == 0:
            f.write("\n")

    f.write("\n};\n\n")

    f.write(
        f"const unsigned int tiny_model_len = "
        f"{len(model_bytes)};\n"
    )

    f.write("\n#endif\n")

print("tiny_model.h created")
print("Model bytes:", len(model_bytes))

In [ ]:
tiny_scaler = scalers["tiny"]

tiny_mean = tiny_scaler.mean_
tiny_scale = tiny_scaler.scale_

print("Tiny scaler mean:")
print(tiny_mean)

print("\nTiny scaler scale:")
print(tiny_scale)

print("\nNumber of features:", len(tiny_mean))

In [ ]:
tiny_scaler = scalers["tiny"]

num_features = len(tiny_scaler.mean_)

with open("tiny_scaler.h", "w") as f:

    f.write("#ifndef TINY_SCALER_H\n")
    f.write("#define TINY_SCALER_H\n\n")

    # Mean
    f.write(f"const float tiny_mean[{num_features}] = {{\n")

    for i, value in enumerate(tiny_scaler.mean_):
        f.write(f"    {value:.10e}f")

        if i < num_features - 1:
            f.write(",")

        f.write("\n")

    f.write("};\n\n")

    # Scale
    f.write(f"const float tiny_scale[{num_features}] = {{\n")

    for i, value in enumerate(tiny_scaler.scale_):
        f.write(f"    {value:.10e}f")

        if i < num_features - 1:
            f.write(",")

        f.write("\n")

    f.write("};\n\n")
    f.write("#endif\n")

print("tiny_scaler.h created")
print("Number of features:", num_features)

In [ ]:
# ============================================================
# PART 2 — Raw-Signal TinyML Architecture Benchmark
# Idea 1: Dense vs CNN vs Quantized CNN vs Depthwise CNN
# Idea 3: Raw Signal vs Engineered Features
# ============================================================

In [ ]:
# ============================================================
# Build RAW IMU dataset
# Shape: (samples, 128, 6)
# ============================================================

X_raw = []
y_raw = []

for label_idx, label in enumerate(class_names):

    windows = all_data[label]

    for window in windows:
        X_raw.append(window)
        y_raw.append(label_idx)

X_raw = np.asarray(X_raw, dtype=np.float32)
y_raw = np.asarray(y_raw, dtype=np.int32)

print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)

print("\nClass distribution:")

for idx, label in enumerate(class_names):
    count = np.sum(y_raw == idx)
    print(f"{label:12s}: {count}")

In [ ]:
# ============================================================
# Apply the SAME train / validation / test split to RAW data
# ============================================================

X_raw_train = X_raw[train_idx]
y_raw_train = y_raw[train_idx]

X_raw_val = X_raw[val_idx]
y_raw_val = y_raw[val_idx]

X_raw_test = X_raw[test_idx]
y_raw_test = y_raw[test_idx]


print("RAW dataset split")
print("-" * 40)

print("Train:", X_raw_train.shape, y_raw_train.shape)
print("Val:  ", X_raw_val.shape, y_raw_val.shape)
print("Test: ", X_raw_test.shape, y_raw_test.shape)


print("\nClass distribution:")

for name, labels in [
    ("Train", y_raw_train),
    ("Val", y_raw_val),
    ("Test", y_raw_test)
]:
    counts = np.bincount(
        labels,
        minlength=len(class_names)
    )

    print(f"{name:5s}:", counts)

In [ ]:
# ============================================================
# Normalize RAW IMU data
# Statistics are computed ONLY from the training set
# ============================================================

raw_mean = X_raw_train.mean(
    axis=(0, 1),
    keepdims=True
)

raw_std = X_raw_train.std(
    axis=(0, 1),
    keepdims=True
)

# Numerical safety
raw_std = np.where(
    raw_std < 1e-8,
    1.0,
    raw_std
)

X_raw_train_scaled = (
    (X_raw_train - raw_mean) / raw_std
).astype(np.float32)

X_raw_val_scaled = (
    (X_raw_val - raw_mean) / raw_std
).astype(np.float32)

X_raw_test_scaled = (
    (X_raw_test - raw_mean) / raw_std
).astype(np.float32)


print("Channel means:")
print(raw_mean.reshape(-1))

print("\nChannel standard deviations:")
print(raw_std.reshape(-1))

print("\nScaled shapes:")
print("Train:", X_raw_train_scaled.shape)
print("Val:  ", X_raw_val_scaled.shape)
print("Test: ", X_raw_test_scaled.shape)

print("\nTraining data after normalization:")
print(
    "Mean:",
    X_raw_train_scaled.mean(axis=(0, 1))
)

print(
    "Std: ",
    X_raw_train_scaled.std(axis=(0, 1))
)

In [ ]:
# ============================================================
# Raw-Signal 1D CNN Baseline
# ============================================================

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense
)


def build_raw_cnn(num_classes=4):

    model = Sequential([
        tf.keras.Input(shape=(WINDOW_SIZE, 6)),

        Conv1D(
            filters=16,
            kernel_size=5,
            padding="same",
            activation="relu"
        ),

        MaxPooling1D(pool_size=2),

        Conv1D(
            filters=32,
            kernel_size=3,
            padding="same",
            activation="relu"
        ),

        GlobalAveragePooling1D(),

        Dense(
            16,
            activation="relu"
        ),

        Dense(
            num_classes,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


raw_cnn = build_raw_cnn(
    num_classes=len(class_names)
)

raw_cnn.summary()

print(
    "\nTotal parameters:",
    raw_cnn.count_params()
)

In [ ]:
# ============================================================
# Train Raw CNN
# ============================================================

tf.keras.utils.set_random_seed(SEED)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

raw_cnn_history = raw_cnn.fit(
    X_raw_train_scaled,
    y_raw_train,

    validation_data=(
        X_raw_val_scaled,
        y_raw_val
    ),

    epochs=150,
    batch_size=16,

    callbacks=[early_stop],

    verbose=0
)

raw_cnn_test_loss, raw_cnn_test_accuracy = raw_cnn.evaluate(
    X_raw_test_scaled,
    y_raw_test,
    verbose=0
)

print(
    "Epochs:",
    len(raw_cnn_history.history["loss"])
)

print(
    f"Test loss: {raw_cnn_test_loss:.4f}"
)

print(
    f"Test accuracy: {raw_cnn_test_accuracy:.4f}"
)

In [ ]:
# ============================================================
# Evaluate Raw CNN
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score
)

raw_cnn_prob = raw_cnn.predict(
    X_raw_test_scaled,
    verbose=0
)

raw_cnn_pred = np.argmax(
    raw_cnn_prob,
    axis=1
)

raw_cnn_f1 = f1_score(
    y_raw_test,
    raw_cnn_pred,
    average="macro"
)

raw_cnn_cm = confusion_matrix(
    y_raw_test,
    raw_cnn_pred
)

print("RAW CNN RESULTS")
print("=" * 50)

print(
    f"Accuracy: {raw_cnn_test_accuracy:.4f}"
)

print(
    f"Macro F1: {raw_cnn_f1:.4f}"
)

print("\nConfusion Matrix:")
print(raw_cnn_cm)

print("\nClassification Report:")

print(
    classification_report(
        y_raw_test,
        raw_cnn_pred,
        target_names=class_names,
        digits=4
    )
)

In [ ]:
# ============================================================
# Convert Raw CNN to Float32 TFLite
# ============================================================

import os
import tensorflow as tf

raw_cnn_converter = tf.lite.TFLiteConverter.from_keras_model(
    raw_cnn
)

raw_cnn_float_tflite = raw_cnn_converter.convert()

raw_cnn_float_filename = "raw_cnn_float32.tflite"

with open(raw_cnn_float_filename, "wb") as f:
    f.write(raw_cnn_float_tflite)

raw_cnn_float_size_kb = (
    os.path.getsize(raw_cnn_float_filename) / 1024
)

print("RAW CNN Float32 TFLite")
print("=" * 40)

print(
    f"Model size: {raw_cnn_float_size_kb:.2f} KB"
)

print(
    f"Parameters: {raw_cnn.count_params()}"
)

In [ ]:
# ============================================================
# Full INT8 Quantization of Raw CNN
# ============================================================

def raw_cnn_representative_dataset():

    for i in range(min(100, len(X_raw_train_scaled))):

        sample = X_raw_train_scaled[i:i+1].astype(
            np.float32
        )

        yield [sample]


raw_cnn_int8_converter = (
    tf.lite.TFLiteConverter.from_keras_model(raw_cnn)
)

raw_cnn_int8_converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

raw_cnn_int8_converter.representative_dataset = (
    raw_cnn_representative_dataset
)

raw_cnn_int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

raw_cnn_int8_converter.inference_input_type = tf.int8
raw_cnn_int8_converter.inference_output_type = tf.int8


raw_cnn_int8_tflite = (
    raw_cnn_int8_converter.convert()
)


raw_cnn_int8_filename = "raw_cnn_int8.tflite"

with open(raw_cnn_int8_filename, "wb") as f:
    f.write(raw_cnn_int8_tflite)


raw_cnn_int8_size_kb = (
    os.path.getsize(raw_cnn_int8_filename) / 1024
)


print("RAW CNN INT8 TFLite")
print("=" * 40)

print(
    f"Float32 size: {raw_cnn_float_size_kb:.2f} KB"
)

print(
    f"INT8 size:    {raw_cnn_int8_size_kb:.2f} KB"
)

print(
    f"Size reduction: "
    f"{100 * (1 - raw_cnn_int8_size_kb / raw_cnn_float_size_kb):.2f}%"
)

In [ ]:
# ============================================================
# Evaluate Raw CNN INT8
# ============================================================

def evaluate_raw_int8(
    model_content,
    X_test,
    y_test
):

    interpreter = tf.lite.Interpreter(
        model_content=model_content
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    input_scale, input_zero_point = (
        input_details["quantization"]
    )

    output_scale, output_zero_point = (
        output_details["quantization"]
    )

    print("Input dtype:", input_details["dtype"])
    print("Output dtype:", output_details["dtype"])

    print(
        "Input quantization:",
        input_scale,
        input_zero_point
    )

    print(
        "Output quantization:",
        output_scale,
        output_zero_point
    )

    predictions = []

    for sample in X_test:

        # Shape -> (1, 128, 6)
        sample = sample[
            np.newaxis, ...
        ].astype(np.float32)

        # Float32 -> INT8
        sample_q = (
            sample / input_scale
            + input_zero_point
        )

        sample_q = np.clip(
            np.round(sample_q),
            -128,
            127
        ).astype(np.int8)

        interpreter.set_tensor(
            input_details["index"],
            sample_q
        )

        interpreter.invoke()

        output_q = interpreter.get_tensor(
            output_details["index"]
        )

        # INT8 -> Float32
        output_float = (
            output_q.astype(np.float32)
            - output_zero_point
        ) * output_scale

        pred = np.argmax(
            output_float[0]
        )

        predictions.append(pred)

    predictions = np.array(predictions)

    accuracy = np.mean(
        predictions == y_test
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="macro"
    )

    return accuracy, f1, predictions

In [ ]:
raw_cnn_int8_accuracy, raw_cnn_int8_f1, raw_cnn_int8_pred = (
    evaluate_raw_int8(
        raw_cnn_int8_tflite,
        X_raw_test_scaled,
        y_raw_test
    )
)

print("\nRAW CNN INT8 RESULTS")
print("=" * 40)

print(
    f"Accuracy: {raw_cnn_int8_accuracy:.4f}"
)

print(
    f"Macro F1: {raw_cnn_int8_f1:.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_raw_test,
        raw_cnn_int8_pred
    )
)

In [ ]:
# ============================================================
# Depthwise-Separable 1D CNN
# ============================================================

from tensorflow.keras.layers import (
    SeparableConv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    Dense
)


def build_raw_depthwise_cnn(num_classes=4):

    model = tf.keras.Sequential([
        tf.keras.Input(shape=(WINDOW_SIZE, 6)),

        SeparableConv1D(
            filters=16,
            kernel_size=5,
            padding="same",
            activation="relu"
        ),

        MaxPooling1D(
            pool_size=2
        ),

        SeparableConv1D(
            filters=32,
            kernel_size=3,
            padding="same",
            activation="relu"
        ),

        GlobalAveragePooling1D(),

        Dense(
            16,
            activation="relu"
        ),

        Dense(
            num_classes,
            activation="softmax"
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


raw_depthwise_cnn = build_raw_depthwise_cnn(
    num_classes=len(class_names)
)

raw_depthwise_cnn.summary()

print(
    "\nTotal parameters:",
    raw_depthwise_cnn.count_params()
)

In [161]:
# ============================================================
# Train Depthwise-Separable CNN
# ============================================================

tf.keras.utils.set_random_seed(SEED)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

raw_depthwise_history = raw_depthwise_cnn.fit(
    X_raw_train_scaled,
    y_raw_train,

    validation_data=(
        X_raw_val_scaled,
        y_raw_val
    ),

    epochs=150,
    batch_size=16,

    callbacks=[early_stop],

    verbose=0
)

raw_depthwise_test_loss, raw_depthwise_test_accuracy = (
    raw_depthwise_cnn.evaluate(
        X_raw_test_scaled,
        y_raw_test,
        verbose=0
    )
)

print(
    "Epochs:",
    len(raw_depthwise_history.history["loss"])
)

print(
    f"Test loss: {raw_depthwise_test_loss:.4f}"
)

print(
    f"Test accuracy: {raw_depthwise_test_accuracy:.4f}"
)

print(
    f"Parameters: {raw_depthwise_cnn.count_params()}"
)

Epochs: 150
Test loss: 0.0000
Test accuracy: 1.0000
Parameters: 1330


In [162]:
# ============================================================
# Evaluate Depthwise-Separable CNN
# ============================================================

raw_depthwise_prob = raw_depthwise_cnn.predict(
    X_raw_test_scaled,
    verbose=0
)

raw_depthwise_pred = np.argmax(
    raw_depthwise_prob,
    axis=1
)

raw_depthwise_f1 = f1_score(
    y_raw_test,
    raw_depthwise_pred,
    average="macro"
)

raw_depthwise_cm = confusion_matrix(
    y_raw_test,
    raw_depthwise_pred
)

print("DEPTHWISE CNN RESULTS")
print("=" * 50)

print(
    f"Accuracy: {raw_depthwise_test_accuracy:.4f}"
)

print(
    f"Macro F1: {raw_depthwise_f1:.4f}"
)

print("\nConfusion Matrix:")
print(raw_depthwise_cm)

print("\nClassification Report:")

print(
    classification_report(
        y_raw_test,
        raw_depthwise_pred,
        target_names=class_names,
        digits=4
    )
)

DEPTHWISE CNN RESULTS
Accuracy: 1.0000
Macro F1: 1.0000

Confusion Matrix:
[[16  0  0  0]
 [ 0 16  0  0]
 [ 0  0 16  0]
 [ 0  0  0 16]]

Classification Report:
              precision    recall  f1-score   support

      circle     1.0000    1.0000    1.0000        16
  left_right     1.0000    1.0000    1.0000        16
        rest     1.0000    1.0000    1.0000        16
     up_down     1.0000    1.0000    1.0000        16

    accuracy                         1.0000        64
   macro avg     1.0000    1.0000    1.0000        64
weighted avg     1.0000    1.0000    1.0000        64



In [163]:
# ============================================================
# Convert Depthwise CNN to Float32 TFLite
# ============================================================

depthwise_float_converter = (
    tf.lite.TFLiteConverter.from_keras_model(
        raw_depthwise_cnn
    )
)

depthwise_float_tflite = (
    depthwise_float_converter.convert()
)

depthwise_float_filename = (
    "raw_depthwise_float32.tflite"
)

with open(depthwise_float_filename, "wb") as f:
    f.write(depthwise_float_tflite)

depthwise_float_size_kb = (
    os.path.getsize(depthwise_float_filename)
    / 1024
)

print("DEPTHWISE CNN Float32 TFLite")
print("=" * 40)

print(
    f"Model size: {depthwise_float_size_kb:.2f} KB"
)

print(
    f"Parameters: {raw_depthwise_cnn.count_params()}"
)

print(
    f"Accuracy: {raw_depthwise_test_accuracy:.4f}"
)

print(
    f"Macro F1: {raw_depthwise_f1:.4f}"
)

Saved artifact at '/tmp/tmposi7pd5w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='keras_tensor_127')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137589016991824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639411344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639404432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639402704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639404240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639402512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639406160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639401360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639403856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639403472: TensorSpec(shape=(), dtype=tf.resource, name=None)
DEPTHWISE CNN Flo

In [164]:
# ============================================================
# Full INT8 Quantization of Depthwise CNN
# ============================================================

def depthwise_representative_dataset():

    for i in range(min(100, len(X_raw_train_scaled))):

        sample = X_raw_train_scaled[
            i:i+1
        ].astype(np.float32)

        yield [sample]


depthwise_int8_converter = (
    tf.lite.TFLiteConverter.from_keras_model(
        raw_depthwise_cnn
    )
)

depthwise_int8_converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

depthwise_int8_converter.representative_dataset = (
    depthwise_representative_dataset
)

depthwise_int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

depthwise_int8_converter.inference_input_type = tf.int8
depthwise_int8_converter.inference_output_type = tf.int8


depthwise_int8_tflite = (
    depthwise_int8_converter.convert()
)


depthwise_int8_filename = (
    "raw_depthwise_int8.tflite"
)

with open(depthwise_int8_filename, "wb") as f:
    f.write(depthwise_int8_tflite)


depthwise_int8_size_kb = (
    os.path.getsize(depthwise_int8_filename)
    / 1024
)


print("DEPTHWISE CNN INT8 TFLite")
print("=" * 40)

print(
    f"Float32 size: {depthwise_float_size_kb:.2f} KB"
)

print(
    f"INT8 size:    {depthwise_int8_size_kb:.2f} KB"
)

print(
    f"Quantization reduction: "
    f"{100 * (1 - depthwise_int8_size_kb / depthwise_float_size_kb):.2f}%"
)

print(
    f"Reduction vs Standard CNN Float32: "
    f"{100 * (1 - depthwise_int8_size_kb / raw_cnn_float_size_kb):.2f}%"
)

Saved artifact at '/tmp/tmpkjn2thee'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 6), dtype=tf.float32, name='keras_tensor_127')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137589016991824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639411344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639404432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639402704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639404240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639402512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639406160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639401360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639403856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137588639403472: TensorSpec(shape=(), dtype=tf.resource, name=None)
DEPTHWISE CNN INT

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [165]:
# ============================================================
# Evaluate Depthwise CNN INT8
# ============================================================

depthwise_int8_accuracy, depthwise_int8_f1, depthwise_int8_pred = (
    evaluate_raw_int8(
        depthwise_int8_tflite,
        X_raw_test_scaled,
        y_raw_test
    )
)

print("\nDEPTHWISE CNN INT8 RESULTS")
print("=" * 40)

print(
    f"Accuracy: {depthwise_int8_accuracy:.4f}"
)

print(
    f"Macro F1: {depthwise_int8_f1:.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_raw_test,
        depthwise_int8_pred
    )
)

Input dtype: <class 'numpy.int8'>
Output dtype: <class 'numpy.int8'>
Input quantization: 0.04253646358847618 11
Output quantization: 0.00390625 -128

DEPTHWISE CNN INT8 RESULTS
Accuracy: 1.0000
Macro F1: 1.0000

Confusion Matrix:
[[16  0  0  0]
 [ 0 16  0  0]
 [ 0  0 16  0]
 [ 0  0  0 16]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [166]:
# ============================================================
# Estimate MACs for Conv1D / SeparableConv1D / Dense models
# ============================================================

def estimate_macs(model):
    total_macs = 0

    for layer in model.layers:

        # --------------------------
        # Standard Conv1D
        # --------------------------
        if isinstance(layer, tf.keras.layers.Conv1D):

            input_shape = layer.input.shape
            output_shape = layer.output.shape

            input_channels = int(input_shape[-1])
            output_length = int(output_shape[1])
            output_channels = int(output_shape[-1])

            kernel_size = int(layer.kernel_size[0])

            macs = (
                output_length
                * output_channels
                * kernel_size
                * input_channels
            )

            total_macs += macs

            print(
                f"{layer.name:25s} "
                f"{macs:,} MACs"
            )

        # --------------------------
        # Depthwise-Separable Conv1D
        # --------------------------
        elif isinstance(
            layer,
            tf.keras.layers.SeparableConv1D
        ):

            input_shape = layer.input.shape
            output_shape = layer.output.shape

            input_channels = int(input_shape[-1])
            output_length = int(output_shape[1])
            output_channels = int(output_shape[-1])

            kernel_size = int(layer.kernel_size[0])

            # Depthwise part
            depthwise_macs = (
                output_length
                * input_channels
                * kernel_size
            )

            # Pointwise 1x1 part
            pointwise_macs = (
                output_length
                * input_channels
                * output_channels
            )

            macs = depthwise_macs + pointwise_macs

            total_macs += macs

            print(
                f"{layer.name:25s} "
                f"{macs:,} MACs "
                f"(DW={depthwise_macs:,}, "
                f"PW={pointwise_macs:,})"
            )

        # --------------------------
        # Dense
        # --------------------------
        elif isinstance(
            layer,
            tf.keras.layers.Dense
        ):

            input_dim = int(layer.input.shape[-1])
            output_dim = int(layer.output.shape[-1])

            macs = input_dim * output_dim

            total_macs += macs

            print(
                f"{layer.name:25s} "
                f"{macs:,} MACs"
            )

    return total_macs

In [167]:
print("STANDARD CNN")
print("=" * 50)

raw_cnn_macs = estimate_macs(raw_cnn)

print(
    f"\nTotal MACs: {raw_cnn_macs:,}"
)


print("\n" + "=" * 50)

print("DEPTHWISE CNN")
print("=" * 50)

raw_depthwise_macs = estimate_macs(
    raw_depthwise_cnn
)

print(
    f"\nTotal MACs: {raw_depthwise_macs:,}"
)


print("\n" + "=" * 50)

mac_reduction = (
    100
    * (
        1
        - raw_depthwise_macs
        / raw_cnn_macs
    )
)

print(
    f"MAC reduction with Depthwise: "
    f"{mac_reduction:.2f}%"
)

STANDARD CNN
conv1d                    61,440 MACs
conv1d_1                  98,304 MACs
dense_90                  512 MACs
dense_91                  64 MACs

Total MACs: 160,320

DEPTHWISE CNN
separable_conv1d          16,128 MACs (DW=3,840, PW=12,288)
separable_conv1d_1        35,840 MACs (DW=3,072, PW=32,768)
dense_92                  512 MACs
dense_93                  64 MACs

Total MACs: 52,544

MAC reduction with Depthwise: 67.23%
